# Async Subagents

In the basics notebook, a supervisor delegated with the **`task`** tool. That call is **synchronous**: the supervisor stops and waits, idle, until the subagent returns its final report.

An **async subagent** runs as a **background job on an [Agent Protocol](https://github.com/langchain-ai/agent-protocol) server**. Launching one returns a **task id immediately** — the supervisor keeps talking to you while the work happens elsewhere. You can then **check**, **update**, or **cancel** the job by id.

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead><tr>
<th style="border:1px solid #999; padding:10px; text-align:left;">&nbsp;</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Sync subagent (<code>task</code>)</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Async subagent</th>
</tr></thead>
<tbody>
<tr><td style="border:1px solid #999; padding:10px;"><b>Launch</b></td><td style="border:1px solid #999; padding:10px;">Blocks until the subagent finishes</td><td style="border:1px solid #999; padding:10px;">Returns a task id immediately</td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Concurrency</b></td><td style="border:1px solid #999; padding:10px;">Parallel, but the supervisor is blocked</td><td style="border:1px solid #999; padding:10px;">Parallel and non-blocking</td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Mid-task updates</b></td><td style="border:1px solid #999; padding:10px;">Not possible</td><td style="border:1px solid #999; padding:10px;"><code>update_async_task</code></td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Cancellation</b></td><td style="border:1px solid #999; padding:10px;">Not possible</td><td style="border:1px solid #999; padding:10px;"><code>cancel_async_task</code></td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Runs</b></td><td style="border:1px solid #999; padding:10px;">In-process, ephemeral</td><td style="border:1px solid #999; padding:10px;">On a server, on its own thread</td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Best for</b></td><td style="border:1px solid #999; padding:10px;">Short work you wait on</td><td style="border:1px solid #999; padding:10px;">Long-running, interactive work</td></tr>
</tbody></table>

> Async subagents are a **preview** feature. They talk to any Agent Protocol server — a LangSmith Deployment or a self-hosted one.

## How it works

The supervisor and the subagent live in **two different processes**:

- The **supervisor** runs right here in the notebook (`create_deep_agent`).
- The **`researcher` subagent** is served by a local **`langgraph dev`** server that speaks the Agent Protocol.

`create_deep_agent` sees a subagent spec that carries a **`graph_id`** and routes it to **`AsyncSubAgentMiddleware`** instead of the usual `task` tool. That middleware adds **five tools** — `start_async_task`, `check_async_task`, `update_async_task`, `cancel_async_task`, `list_async_tasks` — and keeps a dedicated **`async_tasks`** channel in the supervisor's state so task ids survive context compaction.

**Transports:** omit `url` and calls route in-process (**ASGI**, for co-deployed graphs). Set `url` and calls go over **HTTP** to a remote (or, as here, a local) server. We use HTTP to `localhost:2024`.

## Step 1 — Start the subagent server

Two files in this repo define the deployment:

- **`async_agents/researcher.py`** — an ordinary compiled deep agent (Tavily search + a travel-research prompt) exposed as `graph`.
- **`langgraph.json`** — registers it as graph id `researcher` and points the server at `.env` for API keys.

In a **separate terminal**, start the server and leave it running:

```bash
uv run langgraph dev --no-browser --n-jobs-per-worker 10
```

`--n-jobs-per-worker 10` widens the worker pool so several background tasks can run at once (the default is 1). The cell below checks that the server is reachable.

In [1]:
%load_ext autoreload
%autoreload 2

In [11]:
from dotenv import load_dotenv
load_dotenv(override=True)

import time
from util import print_last_exchange
from langgraph.checkpoint.memory import MemorySaver
from langgraph_sdk import get_sync_client
from deepagents import AsyncSubAgent, create_deep_agent

model = "claude-haiku-4-5-20251001"
SERVER_URL = "http://localhost:2024"

In [12]:
# The server must be running (see Step 1). This confirms it's up and the graph is registered.
client = get_sync_client(url=SERVER_URL)
print("Registered graphs:", [a["graph_id"] for a in client.assistants.search()])

Registered graphs: ['researcher']


## Step 2 — Wire the supervisor to the async subagent

An `AsyncSubAgent` is just a spec: a `name`, a `description` the supervisor uses to decide when to delegate, the `graph_id` on the server, and an optional `url` (present here, so HTTP transport).

We give the supervisor a **checkpointer + `thread_id`** so its `async_tasks` channel persists across turns — that's how it remembers task ids between our messages.

In [15]:
researcher = AsyncSubAgent(
    name="researcher",
    description=(
        "Researches a travel destination or question using web search. "
        "Use for questions that need multiple searches and synthesis."
    ),
    graph_id="researcher",   # must match a graph in langgraph.json
    url=SERVER_URL,          # url present -> HTTP transport to the server
)

supervisor = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a travel-planning supervisor. Delegate research to your async "
        "`researcher` subagent, and relay its findings back to the user."
    ),
    subagents=[researcher],
    checkpointer=MemorySaver(),   # persists the async_tasks channel across turns
)

config = {"configurable": {"thread_id": "trip-planning"}}

In [16]:
# Because the spec carried a graph_id, create_deep_agent wired AsyncSubAgentMiddleware,
# which contributes these five tools for managing background work:
from deepagents import AsyncSubAgentMiddleware

for t in AsyncSubAgentMiddleware(async_subagents=[researcher]).tools:
    print("-", t.name)

- start_async_task
- check_async_task
- update_async_task
- cancel_async_task
- list_async_tasks


## Launch a background task

We ask the supervisor to kick off research **in the background**. Notice the reply comes back right away with a **task id** — the supervisor does *not* sit and wait for the research, and it won't poll on its own.

In [6]:
result = supervisor.invoke({"messages": [{"role": "user", "content": (
    "Research the best time of year to visit Kyoto, Japan. "
    "Kick it off in the background and just give me the task id."
)}]}, config=config)
print_last_exchange(result)

### Human

Research the best time of year to visit Kyoto, Japan. Kick it off in the background and just give me the task id.

### Ai

Task ID: `019f2654-9c8b-7cb1-823e-c84532f4a58f`

The research is running in the background. Let me know when you'd like me to check the results.

### The task lives in the `async_tasks` state channel

The task id isn't only buried in a tool message — the middleware records it in a dedicated state channel, so it survives even when the conversation history is summarized.

In [7]:
def show_tasks():
    tasks = supervisor.get_state(config).values.get("async_tasks", {})
    for tid, t in tasks.items():
        print(f"{tid}  status={t['status']:9}  agent={t['agent_name']}")
    return tasks

tasks = show_tasks()

019f2654-9c8b-7cb1-823e-c84532f4a58f  status=running    agent=researcher


## Run several at once

Because launching is non-blocking, the supervisor can fan out **multiple** background jobs from a single turn.

In [8]:
result = supervisor.invoke({"messages": [{"role": "user", "content": (
    "Also research two more things in the background: "
    "(1) which neighborhoods to stay in when visiting Kyoto, and "
    "(2) good day trips from Kyoto. Launch both and report the task ids."
)}]}, config=config)
print_last_exchange(result)
tasks = show_tasks()

### Human

Also research two more things in the background: (1) which neighborhoods to stay in when visiting Kyoto, and (2) good day trips from Kyoto. Launch both and report the task ids.

### Ai

Task IDs:

1. **Neighborhoods**: `019f2654-abf7-7df3-ba1e-b84b1774ad0f`
2. **Day trips**: `019f2654-abf8-7c01-a0ce-1ca4543f43f2`

Both are running in the background. Let me know when you'd like the results.

019f2654-9c8b-7cb1-823e-c84532f4a58f  status=running    agent=researcher
019f2654-abf7-7df3-ba1e-b84b1774ad0f  status=running    agent=researcher
019f2654-abf8-7c01-a0ce-1ca4543f43f2  status=running    agent=researcher


## Check status — live

`list_async_tasks` fetches the **current** status of every tracked task from the server (statuses saved in the chat history are always considered stale).

The background jobs take a little while, so we first poll the server directly until they finish — purely so this notebook renders completed results. In normal use you'd just carry on with other work and ask for an update whenever you like.

In [9]:
def wait_all(task_map, timeout=240):
    deadline = time.time() + timeout
    for t in task_map.values():
        while time.time() < deadline:
            run = client.runs.get(thread_id=t["thread_id"], run_id=t["run_id"])
            if run["status"] != "running":
                break
            time.sleep(3)

wait_all(tasks)

result = supervisor.invoke({"messages": [{"role": "user", "content":
    "List all my research tasks and their current status."
}]}, config=config)
print_last_exchange(result)

### Human

List all my research tasks and their current status.

### Ai

All three tasks are complete:

1. **Best time to visit Kyoto** (`019f2654-9c8b-7cb1-823e-c84532f4a58f`): **success**
2. **Neighborhoods to stay in** (`019f2654-abf7-7df3-ba1e-b84b1774ad0f`): **success**
3. **Day trips from Kyoto** (`019f2654-abf8-7c01-a0ce-1ca4543f43f2`): **success**

Ready to retrieve the results whenever you'd like.

## Collect a result

When a task has succeeded, `check_async_task` returns its final output. We reference the first task (the Kyoto-timing question) by its id.

In [10]:
kyoto_task = list(tasks)[0]

result = supervisor.invoke({"messages": [{"role": "user", "content":
    f"Give me the findings from task {kyoto_task}."
}]}, config=config)
print_last_exchange(result)

### Human

Give me the findings from task 019f2654-9c8b-7cb1-823e-c84532f4a58f.

### Ai

# **KYOTO TRAVEL GUIDE: BEST TIME TO VISIT**

## **SEASONAL WEATHER CONDITIONS**

**Spring (March–May)**
- Temperature: 10–25°C (50–77°F), warming progressively
- Humidity: Low to moderate
- Rainfall: Minimal except late May
- Conditions: Clear, sunny days; ideal for outdoor exploration

**Summer (June–August)**
- Temperature: 28–35°C (82–95°F), frequently exceeding 86°F
- Humidity: Extreme (60–80%), making heat feel more intense
- Rainfall: June–mid-July is tsuyu (rainy season) with intermittent rain; brief intense downpours in August
- Conditions: Oppressive heat; challenging for sightseeing

**Fall (October–November)**
- Temperature: 15–25°C (59–77°F), cooling through season
- Humidity: Low (40–60%)
- Rainfall: Minimal
- Conditions: Clear skies, crisp air; most comfortable weather of the year

**Winter (December–February)**
- Temperature: 0–10°C (32–50°F)
- Humidity: Low
- Rainfall: Occasional but light
- Conditions: Cold but generally dry; can be damp. Most tourist-free period

---

## **PEAK TOURIST SEASONS & CROWD LEVELS**

**Extremely Crowded (High Season)**
- **Late March–Early May (Cherry Blossoms)**: Peak crowds; hotels book months in advance
- **Late October–November (Fall Foliage/Koyō)**: Equally congested; equally expensive
- **Golden Week (late April–early May)**: Japanese national holiday period; additional surge
- **Obon (mid-August)**: Japanese holiday; increased domestic tourism

**Moderate Crowds**
- **June–July**: Rainy season deters some tourists; remainder face humidity
- **September**: Typhoon risk keeps visitors away
- **December–February**: Significantly fewer tourists overall; December gets busier mid-month

**Least Crowded**
- **January–February**: Post-holiday period; quietest months
- **Early September**: Pre-typhoon window; relatively uncrowded

---

## **SEASONAL ATTRACTIONS & FESTIVALS**

| Season | Key Attractions | Notable Festivals |
|--------|-----------------|-------------------|
| **Spring** | Cherry blossoms (Sakura) at temples and gardens; plum blossoms early March | Hirano Shrine Plum Blossom Festival (Feb–Mar) |
| **Summer** | Gion Matsuri (July); fireworks festivals; temple night illuminations | Gion Festival (July 1–31); Arashiyama Nativity (Aug) |
| **Fall** | Autumn foliage (Koyō) at Tofuku-ji, Eikando, Arashiyama; temple illuminations | Moon-viewing festivals; Arashiyama Bamboo Grove transforms with colors |
| **Winter** | Bare temple grounds; winter gardens; illuminated temples (December); New Year preparations | New Year celebrations; winter light displays |

---

## **COST CONSIDERATIONS BY SEASON**

**Expensive**
- Cherry blossom season (March–April): Hotels 2–3× normal rates; restaurants premium pricing; advance bookings essential
- Fall foliage (Oct–Nov): Near cherry blossom pricing
- Golden Week & Obon: Peak holiday rates apply

**Moderate**
- December (except mid-Dec): Moderate pricing; increases toward holidays
- Late May–early June: Pre-rainy season, fewer crowds, reasonable prices

**Most Affordable**
- **January–February**: Lowest accommodation and activity prices; winter deals available
- **September**: Typhoon risk reduces demand; best bargain window
- **Summer (June–August)**: Heat and humidity lower demand; summer discounts available

---

## **RECOMMENDATIONS FOR DIFFERENT TRAVELERS**

**Best for Photography & Nature Lovers**: October–November (Fall)
- Spectacular foliage; clear skies; excellent light
- Trade-off: High prices, very crowded
- Alternative: Late January–February (quiet, clear days, fewer photo competitors)

**Best for Mild Weather & Lower Costs**: March–early April (Early Spring)
- Pleasant temperatures before peak cherry blossom rush
- Some blossoms appear late March
- Fewer crowds than peak season

**Best for Budget-Conscious Travelers**: January–February
- Lowest prices across all categories
- Peaceful temple visits
- Trade-off: Cold weather; fewer seasonal events

**Best for Festival Lovers**: July (Gion Matsuri)
- One of Japan's most iconic festivals
- Trade-off: Extreme heat and humidity; packed crowds

**Best for Active Sightseers**: November
- Comfortable temperatures for long days of walking
- Fall foliage visible throughout season
- Trade-off: Premium pricing; very crowded

---

## **FINAL VERDICT**

| **Priority** | **Recommendation** |
|-------------|-------------------|
| **Best overall experience** | **November** – Perfect weather + world-class foliage + manageable conditions |
| **Best value + experience** | **Late February to early March** – Emerging spring beauty + dramatically lower costs |
| **Budget travel** | **January–early February** – Lowest prices, authentic experience |
| **Specific experiences** | Cherry blossoms = early April; Gion Festival = July; Winter peace = January |

**Bottom line**: Visit **November** if budget allows—the combination of comfortable weather, breathtaking foliage, and festival atmosphere is unmatched. Choose **early March** for a sweet spot of warming weather, emerging spring, and pre-peak season pricing. Reserve **January–February** for budget travel and authentic, uncrowded exploration.

## Send follow-up instructions

`update_async_task` interrupts the task's current run and starts a fresh one **on the same thread** — so the subagent sees the full history plus your new message. The **task id stays the same**; only the internal run id changes.

In [11]:
result = supervisor.invoke({"messages": [{"role": "user", "content":
    f"Send a follow-up to task {kyoto_task}: also give a rough daily budget "
    "in USD for a mid-range traveler."
}]}, config=config)
print_last_exchange(result)

### Human

Send a follow-up to task 019f2654-9c8b-7cb1-823e-c84532f4a58f: also give a rough daily budget in USD for a mid-range traveler.

### Ai

Follow-up sent to task `019f2654-9c8b-7cb1-823e-c84532f4a58f`. The researcher will incorporate the daily budget breakdown and return the updated results. Let me know when you'd like me to check for the updated findings.

In [12]:
# Wait for the updated run, then collect the revised findings.
tasks = supervisor.get_state(config).values.get("async_tasks", {})
wait_all({kyoto_task: tasks[kyoto_task]})

result = supervisor.invoke({"messages": [{"role": "user", "content":
    f"Show me the updated findings from task {kyoto_task}, including the budget."
}]}, config=config)
print_last_exchange(result)

### Human

Show me the updated findings from task 019f2654-9c8b-7cb1-823e-c84532f4a58f, including the budget.

### Ai

# **KYOTO DAILY BUDGET BREAKDOWN: MID-RANGE TRAVELER**

## **SEASON CATEGORIES**

**Peak Season**: March–May, October–November  
**Shoulder Season**: December–February, June–July  
**Low Season**: August–September

---

## **DETAILED COST BREAKDOWN BY CATEGORY**

### **1. ACCOMMODATION (Mid-Range Hotel/Ryokan)**

| Season | Type | Nightly Cost | Notes |
|--------|------|--------------|-------|
| **Peak** | Mid-range hotel | $110–180 USD | Cherry blossom/fall foliage seasons; advance booking essential |
| **Peak** | Mid-range ryokan (no meals) | $100–150 USD | Traditional inn without kaiseki dinner included |
| **Shoulder** | Mid-range hotel | $70–110 USD | December (except mid-month), Feb, June–July |
| **Low** | Mid-range hotel | $50–85 USD | January, August–September; best rates |

**Mid-range standard assumption**: ¥12,000–18,000 ($80–140 USD) per night for a comfortable 3-star hotel or casual ryokan without meals.

---

### **2. MEALS (Daily Total)**

#### **Breakfast**
- Convenience store: $3.50–5.50
- Hotel breakfast included: $0
- Local café breakfast set: $7–10
**Average**: $5–7/day

#### **Lunch**
- Casual ramen or donburi: $7–10
- Sushi counter or casual restaurant: $13–17
- Convenience store bento: $5–8
**Average**: $10–13/day

#### **Dinner**
- Casual izakaya: $13–20
- Nicer restaurant: $27–40
- Mid-range restaurant: $17–27
- Convenience store: $7–10
**Average (mix of casual & nicer)**: $18–25/day

#### **Snacks & Drinks**
- Coffee/tea & snacks: $5–10/day

**Daily Meal Total**: $38–55 USD (Budget option: $25–35; Luxury option: $60–100)

---

### **3. TEMPLE & ATTRACTION ENTRANCE FEES**

| Attraction | Cost |
|-----------|------|
| Fushimi Inari Shrine | Free |
| Kiyomizu-dera | $4 |
| Kinkaku-ji (Golden Pavilion) | $2.75 |
| Ryoan-ji | $3.50 |
| Tofuku-ji | $4 |
| Arashiyama Bamboo Grove | Free |
| Tenryu-ji Temple | $5.50 |
| Ginkaku-ji (Silver Pavilion) | $3.50 |
| Nijo Castle | $9 |
| Philosopher's Path | Free |
| Museum visits | $5–7 |

**Typical Daily Visitation**: 2–3 temples/attractions  
**Average Daily Cost**: $10–15 USD

---

### **4. LOCAL TRANSPORTATION**

**Individual Fares**
- Single bus/subway ride: $1.50–1.75
- IC Card (ICOCA): $13.50 deposit for ~$12 usable credit

**Day Passes**
- Subway & Bus 1-Day Pass: $7.50 (unlimited rides; break-even at 5+ rides)
- Keihan Railway 1-Day Pass: $5

**Typical Daily Transportation**
- Light sightseeing (walking): $0–3
- Moderate sightseeing (3–4 rides): $5–8
- Heavy sightseeing (day pass): $7.50–10

**Average Daily Cost**: $6–10 USD

---

### **5. MISCELLANEOUS (Shopping, Activities, Entertainment)**

| Item | Cost |
|------|------|
| Souvenirs (modest) | $5–20/day |
| Traditional crafts | $15–50+ |
| Geisha district evening | $15–25 |
| Tea ceremony | $20–40 |
| Kimono rental | $40–80 |
| Cooking class | $50–80 |
| Traditional theater | $15–50 |
| Nightlife/karaoke | $20–50 |

**Average Daily Misc** (casual shopping, no special activities): $10–20 USD

---

## **COMPLETE DAILY BUDGET SUMMARY**

### **PEAK SEASON (March–May, October–November)**

| Category | Cost |
|----------|------|
| Accommodation | $110–180 |
| Meals | $45–55 |
| Temples/Attractions | $12–18 |
| Transportation | $8–10 |
| Miscellaneous | $15–25 |
| **TOTAL PER DAY** | **$190–288** |
| **Mid-point** | **$225/day** |

**Weekly Total**: $1,330–2,015  
**10-day trip**: $1,900–2,880

---

### **SHOULDER SEASON (December–February, June–July)**

| Category | Cost |
|----------|------|
| Accommodation | $70–110 |
| Meals | $40–50 |
| Temples/Attractions | $12–18 |
| Transportation | $8–10 |
| Miscellaneous | $12–20 |
| **TOTAL PER DAY** | **$142–208** |
| **Mid-point** | **$155/day** |

**Weekly Total**: $995–1,455  
**10-day trip**: $1,420–2,080

---

### **LOW SEASON (August–September)**

| Category | Cost |
|----------|------|
| Accommodation | $50–85 |
| Meals | $40–50 |
| Temples/Attractions | $12–18 |
| Transportation | $8–10 |
| Miscellaneous | $10–15 |
| **TOTAL PER DAY** | **$120–178** |
| **Mid-point** | **$130/day** |

**Weekly Total**: $840–1,245  
**10-day trip**: $1,200–1,780

---

## **BUDGET SCENARIOS BY TRAVELER TYPE**

### **Budget Traveler**
- Total: $89–130/day ($620–910/week)

### **Mid-Range Traveler (Standard)**
- **Low Season**: $130–155/day
- **Shoulder Season**: $155–200/day
- **Peak Season**: $220–250/day

### **Luxury Traveler**
- Total: $325–600/day ($2,275–4,200/week)

---

## **SEASONAL COST MULTIPLIERS**

| Season | Accommodation Multiplier | Overall Cost Ratio |
|--------|--------------------------|-------------------|
| Peak (Nov, Apr) | 2.2–2.8× low season | 1.7–2.2× |
| Shoulder (Feb, Jul) | 1.4–1.6× low season | 1.2–1.4× |
| Low (Aug, Sept) | 1.0× (baseline) | 1.0× |

---

## **SAMPLE TRIP BUDGETS**

### **5-Day Peak Season (November)**
- Total: $1,200

### **7-Day Low Season (February)**
- Total: $1,015

### **10-Day Shoulder Season (June) – Budget Option**
- Total: $1,340

---

## **KEY TAKEAWAYS**

✅ **Best Value**: August–September ($130–155/day)  
✅ **Sweet Spot**: February–early March ($135–160/day)  
✅ **Peak Season Tax**: Add $70–100/day vs. low season  
✅ **Accommodation Dominates**: 60% of daily budget in peak season  
✅ **Money-Saving Tips**: Walk when possible, lunch sets over dinner, use IC Card strategically

## Cancel a task

`cancel_async_task` stops a running job you no longer need.

In [13]:
# Launch one more...
result = supervisor.invoke({"messages": [{"role": "user", "content":
    "Start a background researcher on the best ryokan (traditional inns) in Hakone; "
    "just give me the task id."
}]}, config=config)
print_last_exchange(result)

### Human

Start a background researcher on the best ryokan (traditional inns) in Hakone; just give me the task id.

### Ai

Task ID: `019f2656-c39c-7950-b371-025500eab98b`

The researcher is running in the background. Let me know when you'd like me to check the results.

In [14]:
# ...then cancel it by id.
hakone_task = list(supervisor.get_state(config).values["async_tasks"])[-1]

result = supervisor.invoke({"messages": [{"role": "user", "content":
    f"Cancel task {hakone_task}."
}]}, config=config)
print_last_exchange(result)

print()
tasks = show_tasks()   # assigned, so the clean table prints without echoing the raw dict

### Human

Cancel task 019f2656-c39c-7950-b371-025500eab98b.

### Ai

Task `019f2656-c39c-7950-b371-025500eab98b` has been cancelled.


019f2654-9c8b-7cb1-823e-c84532f4a58f  status=success    agent=researcher
019f2654-abf7-7df3-ba1e-b84b1774ad0f  status=success    agent=researcher
019f2654-abf8-7c01-a0ce-1ca4543f43f2  status=success    agent=researcher
019f2656-c39c-7950-b371-025500eab98b  status=cancelled  agent=researcher


## Recap

- **Async subagents run in the background** on an Agent Protocol server. Launching returns a task id immediately; the supervisor stays responsive.
- Five tools manage the lifecycle: **`start` / `check` / `update` / `cancel` / `list`**.
- Task metadata lives in the **`async_tasks`** state channel, so ids survive context compaction — pair the supervisor with a **checkpointer + `thread_id`**.
- **Transport** is chosen by the spec: **omit `url`** for in-process **ASGI** (co-deployed graphs), **set `url`** for **HTTP** to a remote server.

**Two rules the middleware's system prompt enforces** (they're load-bearing):

1. After launching, return control to the user — never auto-check or poll in a loop.
2. Statuses in the chat history are stale; always `check`/`list` to get the live status before reporting it.

**When to reach for async over the sync `task` tool:** long-running work, jobs that belong on a specialized remote deployment, or many concurrent tasks whose results you collect later. If a task exhausts the worker pool and launches start queuing, raise `--n-jobs-per-worker`.